# 🧠 COLAB RUNNER: QUANTA / MEIS / NIB / T-FAN IMPLEMENTATION

In [ ]:
# ===========================================
# 📦 ENVIRONMENT SETUP
# ===========================================
# @title Initialize Environment
!pip install torch jax numpy pyyaml faiss-cpu chromadb mlflow --quiet

import os, sys, yaml, torch, random, json
from pathlib import Path

# Ensure core_system is visible
repo_root = Path(".").resolve()
sys.path.append(str(repo_root))

# Seed control
torch.manual_seed(42)
random.seed(42)
print("✅ Environment initialized.")


In [ ]:
# ===========================================
# ⚙️ LOAD CONFIGURATION
# ===========================================
# @title Load Configs
from core_system.utils.config_parser import load_config

cfg_path = "core_system/configs/default.yaml"
config = load_config(cfg_path)

print("Loaded config:")
print(json.dumps(config, indent=2))


In [ ]:
# ===========================================
# 🧩 MODULE IMPORTS
# ===========================================
# @title Import Core Modules
from core_system.tfan import TFAN
from core_system.nib_loop import NIBLoop
from core_system.antifragility import AntifragilityTester
from core_system.meis_orchestrator import MEISOrchestrator
from core_system.utils.logging import log_event

print("✅ All core modules imported successfully.")


In [ ]:
# ===========================================
# 🧠 INITIALIZE MODULES
# ===========================================
# @title Initialize MEIS System
tfan = TFAN(config)
nib = NIBLoop(config)
antifragility = AntifragilityTester(config)
orchestrator = MEISOrchestrator(modules=[tfan, nib])

print("✅ MEIS orchestrator initialized with TFAN + NIBLoop modules.")


In [ ]:
# ===========================================
# 🧪 SYNTHETIC TEST INPUT
# ===========================================
# @title Create Synthetic Input
from core_system.utils.loaders import load_synthetic_data

x, y = load_synthetic_data(size=4, dim=10)
print("Sample input:", x.shape)


In [ ]:
# ===========================================
# 🚀 RUN FORWARD PASS
# ===========================================
# @title Run Forward Pass Through System
results = orchestrator.run(x)
print("Pipeline results:")
for i, r in enumerate(results):
    print(f"Step {i}: keys={list(r.keys())}")


In [ ]:
# ===========================================
# 🧬 ANTIFRAGILITY TEST LOOP
# ===========================================
# @title Run Antifragility Test
noise_levels = [0.01, 0.05, 0.1, 0.2]
for nl in noise_levels:
    metrics = antifragility.run_with_noise(orchestrator, nl)
    print(f"Noise={nl:.2f} → Metrics={metrics}")
    log_event("results/logs/antifragility.log", metrics)


In [ ]:
# ===========================================
# 📊 TOPOLOGY + BETTI PLACEHOLDER TEST
# ===========================================
# @title Test Topological Metrics
from core_system.utils.topology import betti_numbers, bottleneck_distance

bn = betti_numbers(x)
dist = bottleneck_distance(0.5, 0.3)
print("Betti numbers:", bn)
print("Bottleneck distance:", dist)


In [ ]:
# ===========================================
# 🧾 SAVE RUN SUMMARY
# ===========================================
# @title Save Run Summary
summary = {
    "config": config,
    "results": antifragility.history,
    "topology": bn,
}
os.makedirs("results/logs", exist_ok=True)
with open("results/logs/run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("✅ Run summary saved → results/logs/run_summary.json")


In [ ]:
# ===========================================
# 📈 (OPTIONAL) SIMPLE PLOT
# ===========================================
# @title Plot Antifragility Metrics
import matplotlib.pyplot as plt

noise = [m["noise"] for m in antifragility.history]
score = [m["score"] for m in antifragility.history]

plt.plot(noise, score, marker="o")
plt.title("Antifragility Score vs Noise Level")
plt.xlabel("Noise Level")
plt.ylabel("Score")
plt.grid(True)
plt.show()


In [ ]:
# ===========================================
# ✅ COMPLETION MESSAGE
# ===========================================
# @title Done
print("🎉 MEIS / T-FAN system ran successfully end-to-end in Colab!")
